In [ ]:
# Notebook 10
# Interactive Dark-Store Location Intelligence Engine

!pip -q install ipywidgets plotly requests

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
import requests
import math

from IPython.display import display, HTML, clear_output
from google.colab import files

print("✓ Libraries ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 37.9 MB/s eta 0:00:00
✓ Libraries ready


In [ ]:
# Upload final Notebook 7–9 outputs

uploaded = files.upload()

Saving market_expansion_strategy.csv to market_expansion_strategy.csv
Saving market_opportunity_scores.csv to market_opportunity_scores.csv
Saving location_opportunity_features.csv to location_opportunity_features.csv


In [ ]:
# Load datasets

locations = pd.read_csv(
    "location_opportunity_features.csv"
)

market_scores = pd.read_csv(
    "market_opportunity_scores.csv"
)

strategy = pd.read_csv(
    "market_expansion_strategy.csv"
)

print("Location records:", locations.shape)
print("Market scores:", market_scores.shape)
print("Strategy markets:", strategy.shape)

Location records: (3232, 36)
Market scores: (48, 14)
Strategy markets: (48, 15)


In [ ]:
# Validate important columns

required_location_cols = [
    "brand",
    "latitude",
    "longitude",
    "assigned_market"
]

required_market_cols = [
    "assigned_market",
    "Demand_Strength",
    "Opportunity_Score"
]

required_strategy_cols = [
    "assigned_market",
    "Strategy"
]

for col in required_location_cols:
    assert col in locations.columns, f"Missing in locations: {col}"

for col in required_market_cols:
    assert col in market_scores.columns, f"Missing in market_scores: {col}"

for col in required_strategy_cols:
    assert col in strategy.columns, f"Missing in strategy: {col}"

print("✓ Required columns verified")

✓ Required columns verified


In [ ]:
# Merge opportunity scores with strategy

market_intelligence = market_scores.merge(
    strategy[
        ["assigned_market", "Strategy"]
    ],
    on="assigned_market",
    how="left"
)

market_intelligence["assigned_market"] = (
    market_intelligence["assigned_market"]
    .astype(str)
    .str.lower()
    .str.strip()
)

locations["assigned_market"] = (
    locations["assigned_market"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print(
    "Markets available:",
    market_intelligence["assigned_market"].nunique()
)

market_intelligence.head()

Markets available: 48


,assigned_market,Opportunity_Score,Balanced,Demand_Focused,Low_Competition,Demand_Strength,Competition_Risk,Population,Existing_Stores,Rank,Rank_Balanced,Rank_Demand,Rank_LowCompetition,Rank_Stability,Strategy
0,coimbatore,74.992844,74.992844,80.214877,72.350008,0.908019,0.104709,3458045.0,26,1,1.0,1.0,4.0,1.732051,EXPAND
1,vellore,74.142026,74.142026,76.558482,73.402457,0.785377,0.027159,3936331.0,7,2,2.0,5.0,2.0,1.732051,EXPAND
2,madurai,73.992670,73.992670,77.816620,72.281403,0.841981,0.077736,3038252.0,9,3,3.0,4.0,5.0,1.000000,EXPAND
3,palakkad,73.530663,73.530663,74.781580,73.525657,0.735849,0.015970,2809934.0,5,4,4.0,8.0,1.0,3.511885,EXPAND
4,nagpur,73.052309,73.052309,78.430708,70.165420,0.903302,0.175774,4653570.0,35,5,5.0,2.0,7.0,2.516611,EXPAND


In [ ]:
# Geographic distance in kilometres

def haversine_km(lat1, lon1, lat2, lon2):

    R = 6371.0088

    lat1 = np.radians(lat1)
    lon1 = np.radians(lon1)
    lat2 = np.radians(lat2)
    lon2 = np.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        np.sin(dlat / 2) ** 2
        +
        np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlon / 2) ** 2
    )

    return (
        2 * R *
        np.arcsin(np.sqrt(a))
    )

print("✓ Distance engine ready")

✓ Distance engine ready


In [ ]:
# Market geographic centres

market_centres = (
    locations
    .groupby("assigned_market")
    .agg(
        market_lat=("latitude", "mean"),
        market_lon=("longitude", "mean")
    )
    .reset_index()
)

market_intelligence = (
    market_intelligence
    .merge(
        market_centres,
        on="assigned_market",
        how="left"
    )
)

market_intelligence = (
    market_intelligence
    .dropna(
        subset=[
            "market_lat",
            "market_lon"
        ]
    )
    .reset_index(drop=True)
)

print(
    "Geocoded intelligence markets:",
    len(market_intelligence)
)

Geocoded intelligence markets: 48


In [ ]:
# Find nearest modelled market

def find_nearest_location(lat, lon):

    df = market_intelligence.copy()

    df["query_distance_km"] = haversine_km(
        lat,
        lon,
        df["market_lat"].values,
        df["market_lon"].values
    )

    nearest = (
        df
        .sort_values("query_distance_km")
        .iloc[0]
        .copy()
    )

    return nearest


print("✓ Market matching engine ready")

✓ Market matching engine ready


In [ ]:
# Analyse dark stores around candidate location

def analyse_local_competition(
    lat,
    lon,
    selected_brand,
    radius_km
):

    df = locations.copy()

    df["distance_km"] = haversine_km(
        lat,
        lon,
        df["latitude"].values,
        df["longitude"].values
    )

    nearby = df[
        df["distance_km"] <= radius_km
    ].copy()

    total = len(nearby)

    same_brand = len(
        nearby[
            nearby["brand"] == selected_brand
        ]
    )

    competitors = (
        total - same_brand
    )

    competitor_ratio = (
        competitors / total
        if total > 0
        else 0
    )

    stats = {
        "total": total,
        "same_brand": same_brand,
        "competitors": competitors,
        "competitor_ratio": competitor_ratio
    }

    return nearby, stats


print("✓ Competition engine ready")

✓ Competition engine ready


In [ ]:
# Candidate site scoring

def calculate_site_score(
    nearest,
    stats
):

    demand = float(
        nearest["Demand_Strength"]
    )

    market_opportunity = float(
        nearest["Opportunity_Score"]
    ) / 100

    local_whitespace = (
        1 - stats["competitor_ratio"]
    )

    # Weighted site intelligence score

    score = (
        0.45 * demand
        +
        0.35 * market_opportunity
        +
        0.20 * local_whitespace
    ) * 100

    return round(
        min(max(score, 0), 100),
        1
    )


print("✓ Site scoring engine ready")

✓ Site scoring engine ready


In [ ]:
# Business recommendation

def get_recommendation(
    score,
    market_strategy,
    competitor_ratio
):

    if score >= 75:

        decision = "STRONG EXPANSION CANDIDATE"

        insight = (
            "Strong demand and commercial opportunity "
            "make this location attractive for dark-store expansion."
        )

    elif score >= 60:

        decision = "CONSIDER"

        insight = (
            "Commercial potential is attractive, but local "
            "conditions should be validated before expansion."
        )

    elif score >= 45:

        decision = "WATCH"

        insight = (
            "The market shows moderate potential. Expansion "
            "should depend on stronger micro-location evidence."
        )

    else:

        decision = "LOW PRIORITY"

        insight = (
            "Current demand and competitive conditions do not "
            "provide a strong expansion case."
        )

    if competitor_ratio >= 0.70:

        insight += (
            " Competitor concentration is particularly high."
        )

    return decision, insight


print("✓ Recommendation engine ready")

✓ Recommendation engine ready


In [ ]:
# Interactive local competition map

def create_competition_map(
    lat,
    lon,
    nearby
):

    brand_colors = {
        "Blinkit": "#F8CB46",
        "Zepto": "#7A288A",
        "Swiggy Instamart": "#FC8019"
    }

    fig = go.Figure()

    for brand, color in brand_colors.items():

        df = nearby[
            nearby["brand"] == brand
        ]

        if df.empty:
            continue

        fig.add_trace(
            go.Scattermap(
                lat=df["latitude"],
                lon=df["longitude"],

                mode="markers",

                marker=dict(
                    size=10,
                    color=color
                ),

                name=brand,

                hovertemplate=(
                    f"<b>{brand}</b><br>"
                    "Latitude: %{lat:.4f}<br>"
                    "Longitude: %{lon:.4f}"
                    "<extra></extra>"
                )
            )
        )

    # Candidate site

    fig.add_trace(
        go.Scattermap(
            lat=[lat],
            lon=[lon],

            mode="markers",

            marker=dict(
                size=20,
                color="#172A46"
            ),

            name="Candidate Site",

            hovertemplate=(
                "<b>CANDIDATE SITE</b>"
                "<extra></extra>"
            )
        )
    )

    fig.update_layout(

        title=dict(
            text="<b>LOCAL COMPETITIVE LANDSCAPE</b>",
            x=0.5
        ),

        map=dict(
            style="open-street-map",

            center=dict(
                lat=lat,
                lon=lon
            ),

            zoom=10
        ),

        height=520,

        legend=dict(
            orientation="h",
            x=0.5,
            xanchor="center"
        ),

        margin=dict(
            t=80,
            l=20,
            r=20,
            b=20
        )
    )

    return fig


print("✓ Competition map ready")

✓ Competition map ready


In [ ]:
# Indian location geocoder

def geocode_india(place_name):

    place_name = place_name.strip()

    if not place_name:
        return []

    url = (
        "https://nominatim.openstreetmap.org/search"
    )

    params = {
        "q": f"{place_name}, India",
        "format": "jsonv2",
        "limit": 8,
        "countrycodes": "in",
        "addressdetails": 1
    }

    headers = {
        "User-Agent":
        "DarkStoreLocationIntelligenceAcademicProject/1.0"
    }

    try:

        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=15
        )

        response.raise_for_status()

        return response.json()

    except Exception as e:

        print(
            "Geocoding error:",
            str(e)
        )

        return []


print("✓ India-wide place search ready")

✓ India-wide place search ready


In [ ]:
# Interface controls

input_mode = widgets.ToggleButtons(
    options=[
        ("Search Place", "place"),
        ("Use Coordinates", "coordinates")
    ],
    value="place",
    description="Input:"
)


place_input = widgets.Text(
    placeholder="Village, town, locality or city...",
    description="Location:",
    layout=widgets.Layout(
        width="520px"
    )
)


search_button = widgets.Button(
    description="SEARCH",
    button_style="info",
    layout=widgets.Layout(
        width="140px"
    )
)


search_results = widgets.Dropdown(
    options=[],
    description="Match:",
    layout=widgets.Layout(
        width="665px"
    )
)


search_status = widgets.HTML()


latitude_input = widgets.FloatText(
    value=19.0760,
    description="Latitude:"
)


longitude_input = widgets.FloatText(
    value=72.8777,
    description="Longitude:"
)


brand_input = widgets.Dropdown(
    options=[
        "Blinkit",
        "Zepto",
        "Swiggy Instamart"
    ],
    value="Blinkit",
    description="Brand:"
)


radius_input = widgets.IntSlider(
    value=5,
    min=1,
    max=15,
    step=1,
    description="Radius:",
    continuous_update=False
)


analyse_button = widgets.Button(
    description="ANALYSE LOCATION",
    button_style="success",
    layout=widgets.Layout(
        width="180px"
    )
)


output = widgets.Output()

print("✓ Controls created")

✓ Controls created


In [ ]:
# Search button logic

def search_place(button):

    place = place_input.value.strip()

    if not place:

        search_status.value = (
            "<span style='color:#E76F51'>"
            "Enter a location."
            "</span>"
        )

        return

    search_status.value = "Searching..."

    results = geocode_india(place)

    if not results:

        search_results.options = []

        search_status.value = (
            "<span style='color:#E76F51'>"
            "No location found."
            "</span>"
        )

        return

    options = []

    for result in results:

        lat = float(result["lat"])
        lon = float(result["lon"])

        name = result[
            "display_name"
        ]

        options.append(
            (
                name,
                (lat, lon, name)
            )
        )

    search_results.options = options

    if options:
        search_results.value = options[0][1]

    search_status.value = (
        "<span style='color:#2A9D8F'>"
        "✓ Location found"
        "</span>"
    )


search_button.on_click(
    search_place
)


def update_coordinates(change):

    if change["new"] is None:
        return

    lat, lon, name = change["new"]

    latitude_input.value = lat
    longitude_input.value = lon


search_results.observe(
    update_coordinates,
    names="value"
)

print("✓ Search interaction connected")

✓ Search interaction connected


In [ ]:
# ============================================================
# CELL 16 — FINAL LOCATION + COMPETITION INTELLIGENCE REPORT
# ============================================================

def run_location_analysis(button):

    with output:

        clear_output(wait=True)

        try:

            # ==================================================
            # 1. GET SEARCHED LOCATION
            # ==================================================

            if input_mode.value == "place":

                if search_results.value is None:

                    display(
                        HTML("""
                        <div style="
                            padding:16px;
                            border:1px solid #E76F51;
                            border-radius:12px;
                        ">
                            <b>Please search and select a location first.</b>
                        </div>
                        """)
                    )

                    return

                lat, lon, full_place_name = search_results.value

            else:

                lat = float(latitude_input.value)
                lon = float(longitude_input.value)

                full_place_name = f"{lat:.5f}, {lon:.5f}"


            lat = float(lat)
            lon = float(lon)

            brand = brand_input.value
            radius = radius_input.value


            # ==================================================
            # 2. SEARCHED LOCATION NAME
            # ==================================================

            # Pachora, Jalgaon, Maharashtra, India -> Pachora
            if input_mode.value == "place":

                searched_location = (
                    full_place_name
                    .split(",")[0]
                    .strip()
                )

            else:

                searched_location = "Coordinate Location"


            # ==================================================
            # 3. RUN INTELLIGENCE ENGINE
            # ==================================================

            nearest = find_nearest_location(
                lat,
                lon
            )

            nearby, stats = analyse_local_competition(
                lat,
                lon,
                brand,
                radius
            )

            score = calculate_site_score(
                nearest,
                stats
            )

            decision, insight = get_recommendation(
                score,
                nearest["Strategy"],
                stats["competitor_ratio"]
            )


            # ==================================================
            # 4. REFERENCE MARKET DATA
            # ==================================================

            distance = float(
                nearest["query_distance_km"]
            )

            reference_market = str(
                nearest["assigned_market"]
            ).title()

            demand_strength = float(
                nearest["Demand_Strength"]
            )

            demand_pct = demand_strength * 100

            opportunity = float(
                nearest["Opportunity_Score"]
            )

            strategy_value = str(
                nearest["Strategy"]
            )


            # ==================================================
            # 5. COMPETITION DATA
            # ==================================================

            total_stores = int(
                stats["total"]
            )

            competitor_stores = int(
                stats["competitors"]
            )

            same_brand_stores = int(
                stats["same_brand"]
            )

            competitor_ratio = float(
                stats["competitor_ratio"]
            )

            whitespace = (
                1 - competitor_ratio
            ) * 100


            # ==================================================
            # 6. MODEL RELIABILITY
            # ==================================================

            if distance <= 10:

                reliability = "HIGH"
                reliability_color = "#2A9D8F"

                reliability_message = (
                    "The searched location is very close to a modeled "
                    "intelligence market, so demand and opportunity "
                    "estimates have relatively strong geographic support."
                )

            elif distance <= 25:

                reliability = "MODERATE"
                reliability_color = "#E9B949"

                reliability_message = (
                    "The searched location is moderately close to the "
                    "reference intelligence market. Local conditions "
                    "should be validated before making an expansion decision."
                )

            else:

                reliability = "LOW"
                reliability_color = "#E76F51"

                reliability_message = (
                    "Demand and market opportunity are being approximated "
                    f"using {reference_market}, which is {distance:.1f} km "
                    "away. Local competition is calculated around the actual "
                    "searched coordinates, but market-level demand should "
                    "be treated as directional."
                )


            # ==================================================
            # 7. SCORE STYLE
            # ==================================================

            if score >= 75:

                accent = "#2A9D8F"
                score_label = "Excellent"

            elif score >= 60:

                accent = "#D9A52E"
                score_label = "Promising"

            elif score >= 45:

                accent = "#E98245"
                score_label = "Moderate"

            else:

                accent = "#D65A5A"
                score_label = "Weak"


            # ==================================================
            # 8. BRAND COUNTS
            # ==================================================

            brand_counts = (
                nearby["brand"]
                .value_counts()
                .to_dict()
            )


            blinkit_count = int(
                brand_counts.get(
                    "Blinkit",
                    0
                )
            )


            zepto_count = int(
                brand_counts.get(
                    "Zepto",
                    0
                )
            )


            swiggy_count = int(
                brand_counts.get(
                    "Swiggy Instamart",
                    0
                )
            )


            # ==================================================
            # 9. BRAND SHARES
            # ==================================================

            if total_stores > 0:

                blinkit_share = (
                    blinkit_count /
                    total_stores *
                    100
                )

                zepto_share = (
                    zepto_count /
                    total_stores *
                    100
                )

                swiggy_share = (
                    swiggy_count /
                    total_stores *
                    100
                )

            else:

                blinkit_share = 0
                zepto_share = 0
                swiggy_share = 0


            # ==================================================
            # 10. DOMINANT BRAND
            # ==================================================

            if total_stores > 0 and len(brand_counts) > 0:

                dominant_brand = max(
                    brand_counts,
                    key=brand_counts.get
                )

                dominant_count = int(
                    brand_counts[dominant_brand]
                )

                dominant_share = (
                    dominant_count /
                    total_stores *
                    100
                )

            else:

                dominant_brand = "No Brand"
                dominant_count = 0
                dominant_share = 0


            # ==================================================
            # 11. COMPETITION LEVEL
            # ==================================================

            if total_stores == 0:

                competition_level = "OPEN"
                competition_color = "#2A9D8F"

                competition_insight = (
                    f"No mapped quick-commerce stores were found within "
                    f"{radius} km of {searched_location}. This suggests "
                    "significant local competitive whitespace. However, "
                    "absence of mapped competitors alone does not guarantee "
                    "sufficient customer demand."
                )

            elif competitor_ratio >= 0.70:

                competition_level = "HIGH"
                competition_color = "#E76F51"

                competition_insight = (
                    "This location operates in a highly contested "
                    "quick-commerce environment. Expansion would require "
                    "strong delivery economics, differentiation and precise "
                    "micro-location selection."
                )

            elif competitor_ratio >= 0.45:

                competition_level = "MODERATE"
                competition_color = "#E9B949"

                competition_insight = (
                    "Competition is meaningful but not overwhelming. "
                    "A strategically positioned dark store may still "
                    "capture attractive local demand."
                )

            else:

                competition_level = "LOW"
                competition_color = "#2A9D8F"

                competition_insight = (
                    "Competitive pressure is relatively limited, creating "
                    "potential whitespace for expansion if local demand "
                    "and delivery economics are sufficient."
                )


            # ==================================================
            # 12. SELECTED BRAND POSITION
            # ==================================================

            if total_stores > 0:

                selected_brand_share = (
                    same_brand_stores /
                    total_stores *
                    100
                )

            else:

                selected_brand_share = 0


            if total_stores == 0:

                brand_position = (
                    f"No mapped quick-commerce stores were detected within "
                    f"{radius} km of {searched_location}."
                )

            elif same_brand_stores == 0:

                brand_position = (
                    f"{brand} currently has no mapped stores within "
                    f"{radius} km, indicating potential geographic whitespace."
                )

            elif selected_brand_share >= 50:

                brand_position = (
                    f"{brand} already has a strong local footprint, "
                    f"representing approximately {selected_brand_share:.0f}% "
                    "of mapped stores."
                )

            else:

                brand_position = (
                    f"{brand} has an existing but non-dominant local "
                    f"presence with approximately {selected_brand_share:.0f}% "
                    "of mapped stores."
                )


            # ==================================================
            # 13. LOCATION INTELLIGENCE REPORT
            # ==================================================

            location_report = f"""

            <style>

            .intel-report {{

                --bg:#ffffff;
                --card:#f7f8fa;
                --text:#172A46;
                --muted:#667085;
                --border:#E5E8ED;

                max-width:1150px;
                margin:20px auto;

                background:var(--bg);
                color:var(--text);

                border:1px solid var(--border);
                border-radius:18px;

                overflow:hidden;

                font-family:
                    Inter,
                    -apple-system,
                    BlinkMacSystemFont,
                    "Segoe UI",
                    sans-serif;

                box-shadow:
                    0 8px 30px rgba(0,0,0,.04);
            }}


            @media (prefers-color-scheme:dark) {{

                .intel-report {{

                    --bg:#191C21;
                    --card:#22262D;
                    --text:#F4F6F8;
                    --muted:#AAB2BF;
                    --border:#353B45;

                    box-shadow:none;
                }}

            }}


            .intel-header {{

                padding:30px;

                border-bottom:
                    1px solid var(--border);
            }}


            .intel-eyebrow {{

                font-size:10px;

                letter-spacing:2px;

                font-weight:700;

                color:var(--muted);
            }}


            .intel-location {{

                margin-top:10px;

                font-size:12px;

                color:var(--muted);
            }}


            .intel-location strong {{

                color:var(--text);
            }}


            .intel-main-row {{

                display:flex;

                justify-content:
                    space-between;

                align-items:flex-end;

                gap:20px;

                margin-top:20px;
            }}


            .intel-market {{

                margin:0;

                font-size:31px;

                font-weight:750;

                color:var(--text);
            }}


            .intel-decision {{

                display:inline-block;

                margin-top:10px;

                padding:6px 12px;

                border-radius:100px;

                border:
                    1px solid {accent};

                color:{accent};

                font-size:11px;

                font-weight:750;
            }}


            .intel-score {{

                text-align:right;
            }}


            .intel-score-number {{

                font-size:46px;

                line-height:1;

                font-weight:800;

                color:{accent};
            }}


            .intel-score-label {{

                margin-top:7px;

                color:var(--muted);

                font-size:10px;
            }}


            .intel-insight {{

                max-width:850px;

                margin-top:17px;

                color:var(--muted);

                line-height:1.65;

                font-size:13px;
            }}


            .reference-market {{

                margin-top:16px;

                display:inline-block;

                padding:8px 12px;

                background:var(--card);

                border:
                    1px solid var(--border);

                border-radius:8px;

                color:var(--muted);

                font-size:11px;
            }}


            .reference-market strong {{

                color:var(--text);
            }}


            .intel-grid {{

                display:grid;

                grid-template-columns:
                    repeat(3,1fr);

                gap:12px;

                padding:22px;
            }}


            .intel-card {{

                background:var(--card);

                border:
                    1px solid var(--border);

                border-radius:12px;

                padding:17px;
            }}


            .intel-label {{

                color:var(--muted);

                font-size:10px;

                font-weight:700;

                letter-spacing:.7px;

                text-transform:uppercase;
            }}


            .intel-value {{

                margin-top:7px;

                font-size:24px;

                font-weight:750;

                color:var(--text);
            }}


            .intel-reliability {{

                margin:
                    0 22px 22px 22px;

                padding:17px;

                border:
                    1px solid {reliability_color};

                border-radius:11px;

                color:var(--muted);

                font-size:12px;

                line-height:1.65;
            }}


            .intel-reliability strong {{

                color:{reliability_color};
            }}


            .intel-footer {{

                padding:15px 22px;

                border-top:
                    1px solid var(--border);

                color:var(--muted);

                font-size:11px;
            }}


            @media(max-width:750px) {{

                .intel-grid {{

                    grid-template-columns:
                        1fr 1fr;
                }}

                .intel-main-row {{

                    align-items:flex-start;

                    flex-direction:column;
                }}

                .intel-score {{

                    text-align:left;
                }}

            }}

            </style>


            <div class="intel-report">


                <div class="intel-header">


                    <div class="intel-eyebrow">

                        LOCATION INTELLIGENCE REPORT

                    </div>


                    <div class="intel-location">

                        SEARCHED LOCATION •

                        <strong>
                            {full_place_name}
                        </strong>

                    </div>


                    <div class="intel-main-row">


                        <div>


                            <!-- IMPORTANT:
                                 SEARCHED LOCATION IS THE HEADING
                            -->

                            <h2 class="intel-market">

                                {searched_location}

                            </h2>


                            <div class="intel-decision">

                                {decision}

                            </div>


                        </div>


                        <div class="intel-score">


                            <div class="
                                intel-score-number
                            ">

                                {score:.1f}

                            </div>


                            <div class="
                                intel-score-label
                            ">

                                PRELIMINARY SITE SCORE / 100

                                &nbsp;•&nbsp;

                                {score_label}

                            </div>


                        </div>


                    </div>


                    <div class="intel-insight">

                        {insight}

                    </div>


                    <div class="reference-market">

                        REFERENCE INTELLIGENCE MARKET:

                        <strong>
                            {reference_market}
                        </strong>

                        &nbsp;•&nbsp;

                        {distance:.1f} km away

                    </div>


                </div>


                <!-- KPI CARDS -->

                <div class="intel-grid">


                    <div class="intel-card">

                        <div class="intel-label">

                            Demand Strength

                        </div>

                        <div class="intel-value">

                            {demand_strength:.2f}

                        </div>

                    </div>


                    <div class="intel-card">

                        <div class="intel-label">

                            Market Opportunity

                        </div>

                        <div class="intel-value">

                            {opportunity:.1f}

                        </div>

                    </div>


                    <div class="intel-card">

                        <div class="intel-label">

                            Reference Strategy

                        </div>

                        <div class="intel-value">

                            {strategy_value}

                        </div>

                    </div>


                    <div class="intel-card">

                        <div class="intel-label">

                            Nearby Stores

                        </div>

                        <div class="intel-value">

                            {total_stores}

                        </div>

                    </div>


                    <div class="intel-card">

                        <div class="intel-label">

                            Competitors

                        </div>

                        <div class="intel-value">

                            {competitor_stores}

                        </div>

                    </div>


                    <div class="intel-card">

                        <div class="intel-label">

                            Local Whitespace

                        </div>

                        <div class="intel-value">

                            {whitespace:.0f}%

                        </div>

                    </div>


                </div>


                <!-- RELIABILITY -->

                <div class="intel-reliability">

                    <strong>

                        {reliability}
                        MODEL-MATCH RELIABILITY

                    </strong>

                    <br><br>

                    {reliability_message}

                </div>


                <!-- FOOTER -->

                <div class="intel-footer">

                    Candidate Brand:
                    <b>{brand}</b>

                    &nbsp;&nbsp; • &nbsp;&nbsp;

                    Radius:
                    <b>{radius} km</b>

                    &nbsp;&nbsp; • &nbsp;&nbsp;

                    Coordinates:

                    <b>
                        {lat:.5f}, {lon:.5f}
                    </b>

                </div>


            </div>
            """


            display(
                HTML(location_report)
            )


            # ==================================================
            # 14. COMPETITION INTELLIGENCE REPORT
            # ==================================================

            competition_report = f"""

            <style>

            .competition-report {{

                --bg:#ffffff;
                --card:#f7f8fa;
                --text:#172A46;
                --muted:#667085;
                --border:#E5E8ED;

                max-width:1150px;

                margin:26px auto;

                background:var(--bg);

                color:var(--text);

                border:
                    1px solid var(--border);

                border-radius:18px;

                overflow:hidden;

                font-family:
                    Inter,
                    -apple-system,
                    BlinkMacSystemFont,
                    "Segoe UI",
                    sans-serif;
            }}


            @media
            (prefers-color-scheme:dark) {{

                .competition-report {{

                    --bg:#191C21;
                    --card:#22262D;
                    --text:#F4F6F8;
                    --muted:#AAB2BF;
                    --border:#353B45;
                }}

            }}


            .competition-header {{

                padding:30px;

                border-bottom:
                    1px solid var(--border);
            }}


            .competition-eyebrow {{

                font-size:10px;

                letter-spacing:2px;

                font-weight:700;

                color:var(--muted);
            }}


            .competition-title-row {{

                display:flex;

                justify-content:
                    space-between;

                align-items:flex-end;

                gap:20px;

                margin-top:18px;
            }}


            .competition-title {{

                margin:0;

                font-size:30px;

                font-weight:750;

                color:var(--text);
            }}


            .pressure-box {{

                text-align:right;
            }}


            .pressure-value {{

                color:{competition_color};

                font-size:31px;

                font-weight:800;
            }}


            .pressure-label {{

                margin-top:4px;

                font-size:10px;

                color:var(--muted);
            }}


            .competition-insight {{

                margin-top:16px;

                max-width:850px;

                color:var(--muted);

                font-size:13px;

                line-height:1.65;
            }}


            .competition-grid {{

                display:grid;

                grid-template-columns:
                    repeat(4,1fr);

                gap:12px;

                padding:22px;
            }}


            .competition-card {{

                background:var(--card);

                border:
                    1px solid var(--border);

                border-radius:12px;

                padding:17px;
            }}


            .competition-label {{

                color:var(--muted);

                font-size:10px;

                font-weight:700;

                letter-spacing:.7px;

                text-transform:uppercase;
            }}


            .competition-value {{

                margin-top:7px;

                color:var(--text);

                font-size:25px;

                font-weight:750;
            }}


            .brand-area {{

                padding:
                    5px 24px 26px 24px;
            }}


            .section-heading {{

                margin-bottom:18px;

                color:var(--muted);

                font-size:10px;

                font-weight:700;

                letter-spacing:1.5px;
            }}


            .brand-row {{

                display:grid;

                grid-template-columns:
                    165px 1fr 85px;

                align-items:center;

                gap:15px;

                margin:17px 0;
            }}


            .brand-name {{

                color:var(--text);

                font-size:12px;

                font-weight:650;
            }}


            .brand-track {{

                width:100%;

                height:10px;

                background:var(--border);

                border-radius:100px;

                overflow:hidden;
            }}


            .brand-fill {{

                height:100%;

                border-radius:100px;
            }}


            .brand-number {{

                color:var(--muted);

                text-align:right;

                font-size:11px;
            }}


            .competition-analysis {{

                display:grid;

                grid-template-columns:
                    1fr 1fr;

                gap:12px;

                margin:
                    0 24px 24px 24px;
            }}


            .analysis-card {{

                padding:18px;

                background:var(--card);

                border:
                    1px solid var(--border);

                border-radius:12px;
            }}


            .analysis-heading {{

                color:var(--muted);

                font-size:10px;

                font-weight:700;

                letter-spacing:1px;

                margin-bottom:10px;
            }}


            .analysis-text {{

                color:var(--text);

                font-size:13px;

                line-height:1.65;
            }}


            .analysis-text strong {{

                color:{competition_color};
            }}


            .competition-footer {{

                padding:15px 24px;

                border-top:
                    1px solid var(--border);

                color:var(--muted);

                font-size:11px;
            }}


            @media(max-width:750px) {{

                .competition-grid {{

                    grid-template-columns:
                        1fr 1fr;
                }}

                .competition-analysis {{

                    grid-template-columns:
                        1fr;
                }}

            }}

            </style>


            <div class="competition-report">


                <div class="competition-header">


                    <div class="
                        competition-eyebrow
                    ">

                        LOCAL COMPETITIVE INTELLIGENCE

                    </div>


                    <div class="
                        competition-title-row
                    ">


                        <h2 class="
                            competition-title
                        ">

                            {searched_location}
                            Competitive Landscape

                        </h2>


                        <div class="
                            pressure-box
                        ">

                            <div class="
                                pressure-value
                            ">

                                {competition_level}

                            </div>


                            <div class="
                                pressure-label
                            ">

                                COMPETITIVE PRESSURE

                            </div>

                        </div>


                    </div>


                    <div class="
                        competition-insight
                    ">

                        {competition_insight}

                    </div>


                </div>


                <!-- COMPETITION KPIs -->

                <div class="
                    competition-grid
                ">


                    <div class="
                        competition-card
                    ">

                        <div class="
                            competition-label
                        ">

                            Nearby Stores

                        </div>

                        <div class="
                            competition-value
                        ">

                            {total_stores}

                        </div>

                    </div>


                    <div class="
                        competition-card
                    ">

                        <div class="
                            competition-label
                        ">

                            Competitor Stores

                        </div>

                        <div class="
                            competition-value
                        ">

                            {competitor_stores}

                        </div>

                    </div>


                    <div class="
                        competition-card
                    ">

                        <div class="
                            competition-label
                        ">

                            Same Brand Stores

                        </div>

                        <div class="
                            competition-value
                        ">

                            {same_brand_stores}

                        </div>

                    </div>


                    <div class="
                        competition-card
                    ">

                        <div class="
                            competition-label
                        ">

                            Competitor Ratio

                        </div>

                        <div
                            class="
                                competition-value
                            "

                            style="
                                color:
                                {competition_color};
                            "
                        >

                            {competitor_ratio:.0%}

                        </div>

                    </div>


                </div>


                <!-- BRAND PRESENCE -->

                <div class="brand-area">


                    <div class="
                        section-heading
                    ">

                        BRAND PRESENCE WITHIN
                        {radius} KM OF
                        {searched_location.upper()}

                    </div>


                    <!-- BLINKIT -->

                    <div class="brand-row">

                        <div class="brand-name">

                            Blinkit

                        </div>


                        <div class="brand-track">

                            <div
                                class="brand-fill"

                                style="
                                    width:{blinkit_share:.1f}%;
                                    background:#F8CB46;
                                "
                            >
                            </div>

                        </div>


                        <div class="brand-number">

                            {blinkit_count}

                            &nbsp;•&nbsp;

                            {blinkit_share:.0f}%

                        </div>

                    </div>


                    <!-- ZEPTO -->

                    <div class="brand-row">

                        <div class="brand-name">

                            Zepto

                        </div>


                        <div class="brand-track">

                            <div
                                class="brand-fill"

                                style="
                                    width:{zepto_share:.1f}%;
                                    background:#7A288A;
                                "
                            >
                            </div>

                        </div>


                        <div class="brand-number">

                            {zepto_count}

                            &nbsp;•&nbsp;

                            {zepto_share:.0f}%

                        </div>

                    </div>


                    <!-- SWIGGY -->

                    <div class="brand-row">

                        <div class="brand-name">

                            Swiggy Instamart

                        </div>


                        <div class="brand-track">

                            <div
                                class="brand-fill"

                                style="
                                    width:{swiggy_share:.1f}%;
                                    background:#FC8019;
                                "
                            >
                            </div>

                        </div>


                        <div class="brand-number">

                            {swiggy_count}

                            &nbsp;•&nbsp;

                            {swiggy_share:.0f}%

                        </div>

                    </div>


                </div>


                <!-- BUSINESS INTERPRETATION -->

                <div class="
                    competition-analysis
                ">


                    <div class="
                        analysis-card
                    ">

                        <div class="
                            analysis-heading
                        ">

                            MARKET LEADER

                        </div>


                        <div class="
                            analysis-text
                        ">

                            <strong>
                                {dominant_brand}
                            </strong>

                            has the strongest mapped
                            local presence with

                            <strong>
                                {dominant_count}
                                stores
                            </strong>,

                            representing approximately

                            <strong>
                                {dominant_share:.0f}%
                            </strong>

                            of mapped stores.

                        </div>

                    </div>


                    <div class="
                        analysis-card
                    ">

                        <div class="
                            analysis-heading
                        ">

                            {brand.upper()} POSITION

                        </div>


                        <div class="
                            analysis-text
                        ">

                            {brand_position}

                        </div>

                    </div>


                    <div class="
                        analysis-card
                    ">

                        <div class="
                            analysis-heading
                        ">

                            LOCAL WHITESPACE

                        </div>


                        <div class="
                            analysis-text
                        ">

                            Estimated competitive
                            whitespace:

                            <strong>
                                {whitespace:.0f}%
                            </strong>.

                            This reflects local mapped
                            competitive structure and
                            should be interpreted
                            alongside demand.

                        </div>

                    </div>


                    <div class="
                        analysis-card
                    ">

                        <div class="
                            analysis-heading
                        ">

                            COMPETITIVE VERDICT

                        </div>


                        <div class="
                            analysis-text
                        ">

                            Local competitive pressure
                            is classified as

                            <strong>
                                {competition_level}
                            </strong>.

                            Reference-market demand
                            strength is

                            <strong>
                                {demand_pct:.0f}%
                            </strong>

                            with an opportunity score
                            of

                            <strong>
                                {opportunity:.1f}
                            </strong>.

                        </div>

                    </div>


                </div>


                <div class="
                    competition-footer
                ">

                    Analysed Location:
                    <b>{searched_location}</b>

                    &nbsp;&nbsp; • &nbsp;&nbsp;

                    Candidate Brand:
                    <b>{brand}</b>

                    &nbsp;&nbsp; • &nbsp;&nbsp;

                    Radius:
                    <b>{radius} km</b>

                </div>


            </div>
            """


            display(
                HTML(competition_report)
            )


            # ==================================================
            # 15. GEOSPATIAL MAP
            # ==================================================

            display(
                HTML(
                    f"""
                    <div style="
                        max-width:1150px;
                        margin:32px auto 8px auto;
                        font-family:
                            Inter,
                            -apple-system,
                            BlinkMacSystemFont,
                            'Segoe UI',
                            sans-serif;
                    ">

                        <div style="
                            font-size:10px;
                            letter-spacing:2px;
                            opacity:.6;
                            font-weight:700;
                        ">

                            GEOSPATIAL VIEW

                        </div>


                        <div style="
                            font-size:23px;
                            font-weight:750;
                            margin-top:7px;
                        ">

                            {searched_location}
                            Local Dark-Store Network

                        </div>


                        <div style="
                            font-size:12px;
                            opacity:.6;
                            margin-top:5px;
                            margin-bottom:15px;
                        ">

                            Candidate site and mapped
                            quick-commerce stores within
                            {radius} km.

                        </div>

                    </div>
                    """
                )
            )


            fig = create_competition_map(
                lat,
                lon,
                nearby
            )

            fig.show()


        # ======================================================
        # ERROR HANDLER
        # ======================================================

        except Exception as e:

            display(
                HTML(
                    f"""
                    <div style="
                        max-width:1100px;
                        padding:18px;
                        margin:15px auto;

                        border:
                            1px solid #E76F51;

                        border-radius:12px;

                        font-family:
                            Inter,
                            sans-serif;
                    ">

                        <b>
                            Analysis Error
                        </b>

                        <br><br>

                        {type(e).__name__}:
                        {str(e)}

                    </div>
                    """
                )
            )


print(
    "✓ Cell 16 — Final Location Intelligence Engine ready"
)

✓ Cell 16 — Final Location Intelligence Engine ready


In [ ]:
# ============================================================
# CELL 16 — FINAL ANALYSIS + REPORT
# ============================================================

def run_location_analysis(button):

    with output:

        clear_output(wait=True)

        try:

            # --------------------------------------------------
            # 1. GET SEARCHED LOCATION
            # --------------------------------------------------

            if input_mode.value == "place":

                if search_results.value is None:
                    display(
                        HTML("""
                        <div style="
                            padding:16px;
                            border:1px solid #E76F51;
                            border-radius:12px;
                        ">
                            <b>Please search and select a location first.</b>
                        </div>
                        """)
                    )
                    return

                lat, lon, full_place_name = search_results.value

                # IMPORTANT:
                # Actual searched place
                searched_location = (
                    str(full_place_name)
                    .split(",")[0]
                    .strip()
                )

            else:

                lat = float(latitude_input.value)
                lon = float(longitude_input.value)

                full_place_name = (
                    f"{lat:.5f}, {lon:.5f}"
                )

                searched_location = "Selected Coordinates"


            lat = float(lat)
            lon = float(lon)

            brand = brand_input.value
            radius = radius_input.value


            # --------------------------------------------------
            # 2. FIND REFERENCE INTELLIGENCE MARKET
            # --------------------------------------------------

            nearest = find_nearest_location(
                lat,
                lon
            )

            # IMPORTANT:
            # This is NOT the searched place.
            # It is only the closest market in our dataset.

            reference_market = str(
                nearest["assigned_market"]
            ).title()

            distance = float(
                nearest["query_distance_km"]
            )


            # --------------------------------------------------
            # 3. LOCAL COMPETITION
            # --------------------------------------------------

            nearby, stats = analyse_local_competition(
                lat,
                lon,
                brand,
                radius
            )


            # --------------------------------------------------
            # 4. CALCULATE SCORE
            # --------------------------------------------------

            score = calculate_site_score(
                nearest,
                stats
            )


            decision, insight = get_recommendation(
                score,
                nearest["Strategy"],
                stats["competitor_ratio"]
            )


            # --------------------------------------------------
            # 5. MARKET INTELLIGENCE
            # --------------------------------------------------

            demand_strength = float(
                nearest["Demand_Strength"]
            )

            opportunity = float(
                nearest["Opportunity_Score"]
            )

            strategy_value = str(
                nearest["Strategy"]
            )


            # --------------------------------------------------
            # 6. COMPETITION METRICS
            # --------------------------------------------------

            total_stores = int(
                stats["total"]
            )

            competitor_stores = int(
                stats["competitors"]
            )

            same_brand_stores = int(
                stats["same_brand"]
            )

            competitor_ratio = float(
                stats["competitor_ratio"]
            )

            whitespace = (
                1 - competitor_ratio
            ) * 100


            # --------------------------------------------------
            # 7. MODEL RELIABILITY
            # --------------------------------------------------

            if distance <= 10:

                reliability = "HIGH"

                reliability_color = "#2A9D8F"

                reliability_text = (
                    "The searched location is very close to the "
                    "reference intelligence market. Market-level "
                    "estimates therefore have strong geographic support."
                )

            elif distance <= 25:

                reliability = "MODERATE"

                reliability_color = "#E9B949"

                reliability_text = (
                    "The searched location is moderately close to the "
                    "reference intelligence market. Local conditions "
                    "should be validated before expansion."
                )

            else:

                reliability = "LOW"

                reliability_color = "#E76F51"

                reliability_text = (
                    f"{searched_location} is not directly represented "
                    "in the market-level intelligence dataset. "
                    f"{reference_market} is being used as the nearest "
                    f"reference market at {distance:.1f} km away. "
                    "Local competition is still calculated using the "
                    "actual searched coordinates."
                )


            # --------------------------------------------------
            # 8. SCORE STYLE
            # --------------------------------------------------

            if score >= 75:

                accent = "#2A9D8F"
                score_label = "Excellent"

            elif score >= 60:

                accent = "#D4A72C"
                score_label = "Promising"

            elif score >= 45:

                accent = "#E98245"
                score_label = "Moderate"

            else:

                accent = "#D65A5A"
                score_label = "Weak"


            # --------------------------------------------------
            # 9. LOCATION REPORT HTML
            # --------------------------------------------------

            location_report = f"""

            <style>

            .intel-report {{

                --bg:#FFFFFF;
                --card:#F7F8FA;
                --text:#172A46;
                --muted:#667085;
                --border:#E3E7ED;

                max-width:1150px;
                margin:20px auto;

                background:var(--bg);
                color:var(--text);

                border:
                    1px solid var(--border);

                border-radius:18px;

                overflow:hidden;

                font-family:
                    Inter,
                    -apple-system,
                    BlinkMacSystemFont,
                    "Segoe UI",
                    sans-serif;

                box-shadow:
                    0 8px 30px
                    rgba(0,0,0,.04);
            }}


            @media (prefers-color-scheme:dark) {{

                .intel-report {{

                    --bg:#191C21;
                    --card:#22262D;
                    --text:#F4F6F8;
                    --muted:#AAB2BF;
                    --border:#353B45;

                    box-shadow:none;
                }}

            }}


            .intel-header {{

                padding:30px;

                border-bottom:
                    1px solid var(--border);
            }}


            .intel-eyebrow {{

                font-size:10px;

                letter-spacing:2px;

                font-weight:700;

                color:var(--muted);
            }}


            .intel-location {{

                margin-top:11px;

                font-size:11px;

                color:var(--muted);
            }}


            .intel-location strong {{

                color:var(--text);
            }}


            .intel-main-row {{

                display:flex;

                justify-content:
                    space-between;

                align-items:flex-end;

                gap:20px;

                margin-top:22px;
            }}


            .intel-market {{

                margin:0;

                font-size:32px;

                font-weight:750;

                color:var(--text);
            }}


            .intel-decision {{

                display:inline-block;

                margin-top:10px;

                padding:
                    6px 12px;

                border-radius:100px;

                border:
                    1px solid {accent};

                color:{accent};

                font-size:11px;

                font-weight:750;
            }}


            .intel-score {{

                text-align:right;
            }}


            .intel-score-number {{

                font-size:46px;

                line-height:1;

                font-weight:800;

                color:{accent};
            }}


            .intel-score-label {{

                margin-top:7px;

                color:var(--muted);

                font-size:10px;
            }}


            .intel-insight {{

                max-width:850px;

                margin-top:18px;

                color:var(--muted);

                line-height:1.65;

                font-size:13px;
            }}


            .reference-box {{

                display:inline-block;

                margin-top:18px;

                padding:
                    9px 12px;

                background:var(--card);

                border:
                    1px solid var(--border);

                border-radius:8px;

                color:var(--muted);

                font-size:11px;
            }}


            .reference-box strong {{

                color:var(--text);
            }}


            .intel-grid {{

                display:grid;

                grid-template-columns:
                    repeat(3,1fr);

                gap:12px;

                padding:22px;
            }}


            .intel-card {{

                background:var(--card);

                border:
                    1px solid var(--border);

                border-radius:12px;

                padding:18px;
            }}


            .intel-label {{

                color:var(--muted);

                font-size:10px;

                font-weight:700;

                letter-spacing:.7px;

                text-transform:uppercase;
            }}


            .intel-value {{

                margin-top:8px;

                font-size:24px;

                font-weight:750;

                color:var(--text);
            }}


            .intel-reliability {{

                margin:
                    0 22px 22px 22px;

                padding:17px;

                border:
                    1px solid {reliability_color};

                border-radius:11px;

                color:var(--muted);

                font-size:12px;

                line-height:1.65;
            }}


            .intel-reliability strong {{

                color:{reliability_color};
            }}


            .intel-footer {{

                padding:
                    15px 22px;

                border-top:
                    1px solid var(--border);

                color:var(--muted);

                font-size:11px;
            }}


            @media(max-width:750px) {{

                .intel-grid {{

                    grid-template-columns:
                        1fr 1fr;
                }}

                .intel-main-row {{

                    flex-direction:column;

                    align-items:flex-start;
                }}

                .intel-score {{

                    text-align:left;
                }}

            }}

            </style>


            <div class="intel-report">


                <!-- =========================
                     HEADER
                ========================== -->

                <div class="intel-header">


                    <div class="intel-eyebrow">

                        LOCATION INTELLIGENCE REPORT

                    </div>


                    <div class="intel-location">

                        SEARCHED LOCATION •

                        <strong>
                            {full_place_name}
                        </strong>

                    </div>


                    <div class="intel-main-row">


                        <div>


                            <!--
                            CRITICAL FIX:

                            This is the ACTUAL searched
                            location.

                            Pachora will display as Pachora.

                            Nashik is NOT used here.
                            -->


                            <h2 class="intel-market">

                                {searched_location}

                            </h2>


                            <div class="intel-decision">

                                {decision}

                            </div>


                        </div>


                        <div class="intel-score">


                            <div class="
                                intel-score-number
                            ">

                                {score:.1f}

                            </div>


                            <div class="
                                intel-score-label
                            ">

                                PRELIMINARY SITE SCORE / 100

                                &nbsp;•&nbsp;

                                {score_label}

                            </div>


                        </div>


                    </div>


                    <div class="intel-insight">

                        {insight}

                    </div>


                    <!-- REFERENCE MARKET -->

                    <div class="reference-box">

                        REFERENCE INTELLIGENCE MARKET:

                        <strong>
                            {reference_market}
                        </strong>

                        &nbsp;•&nbsp;

                        {distance:.1f} KM AWAY

                    </div>


                </div>


                <!-- =========================
                     KPI GRID
                ========================== -->

                <div class="intel-grid">


                    <div class="intel-card">

                        <div class="intel-label">

                            Demand Strength

                        </div>


                        <div class="intel-value">

                            {demand_strength:.2f}

                        </div>

                    </div>


                    <div class="intel-card">

                        <div class="intel-label">

                            Market Opportunity

                        </div>


                        <div class="intel-value">

                            {opportunity:.1f}

                        </div>

                    </div>


                    <div class="intel-card">

                        <div class="intel-label">

                            Reference Strategy

                        </div>


                        <div class="intel-value">

                            {strategy_value}

                        </div>

                    </div>


                    <div class="intel-card">

                        <div class="intel-label">

                            Nearby Stores

                        </div>


                        <div class="intel-value">

                            {total_stores}

                        </div>

                    </div>


                    <div class="intel-card">

                        <div class="intel-label">

                            Competitors

                        </div>


                        <div class="intel-value">

                            {competitor_stores}

                        </div>

                    </div>


                    <div class="intel-card">

                        <div class="intel-label">

                            Local Whitespace

                        </div>


                        <div class="intel-value">

                            {whitespace:.0f}%

                        </div>

                    </div>


                </div>


                <!-- =========================
                     RELIABILITY
                ========================== -->

                <div class="intel-reliability">


                    <strong>

                        {reliability}
                        MODEL-MATCH RELIABILITY

                    </strong>


                    <br><br>


                    {reliability_text}


                </div>


                <!-- =========================
                     FOOTER
                ========================== -->

                <div class="intel-footer">


                    Candidate Brand:

                    <b>
                        {brand}
                    </b>


                    &nbsp;&nbsp; • &nbsp;&nbsp;


                    Search Radius:

                    <b>
                        {radius} km
                    </b>


                    &nbsp;&nbsp; • &nbsp;&nbsp;


                    Coordinates:

                    <b>

                        {lat:.5f},
                        {lon:.5f}

                    </b>


                </div>


            </div>

            """


            # --------------------------------------------------
            # 10. DISPLAY LOCATION REPORT
            # --------------------------------------------------

            display(
                HTML(location_report)
            )


            # --------------------------------------------------
            # 11. COMPETITION REPORT VARIABLES
            # --------------------------------------------------

            brand_counts = (
                nearby["brand"]
                .value_counts()
                .to_dict()
            )


            blinkit_count = int(
                brand_counts.get(
                    "Blinkit",
                    0
                )
            )


            zepto_count = int(
                brand_counts.get(
                    "Zepto",
                    0
                )
            )


            swiggy_count = int(
                brand_counts.get(
                    "Swiggy Instamart",
                    0
                )
            )


            if total_stores > 0:

                blinkit_share = (
                    blinkit_count /
                    total_stores *
                    100
                )

                zepto_share = (
                    zepto_count /
                    total_stores *
                    100
                )

                swiggy_share = (
                    swiggy_count /
                    total_stores *
                    100
                )

            else:

                blinkit_share = 0
                zepto_share = 0
                swiggy_share = 0


            # --------------------------------------------------
            # 12. COMPETITION LEVEL
            # --------------------------------------------------

            if total_stores == 0:

                competition_level = "OPEN"

                competition_color = "#2A9D8F"

                competition_message = (
                    f"No mapped quick-commerce dark stores were "
                    f"identified within {radius} km of "
                    f"{searched_location}. This indicates substantial "
                    "competitive whitespace, but local demand must still "
                    "be validated."
                )

            elif competitor_ratio >= 0.70:

                competition_level = "HIGH"

                competition_color = "#E76F51"

                competition_message = (
                    "The selected location operates within a highly "
                    "competitive quick-commerce environment. Expansion "
                    "would require strong differentiation and delivery "
                    "economics."
                )

            elif competitor_ratio >= 0.45:

                competition_level = "MODERATE"

                competition_color = "#D4A72C"

                competition_message = (
                    "The location contains meaningful competitive "
                    "activity, but sufficient whitespace may remain "
                    "for a strategically positioned dark store."
                )

            else:

                competition_level = "LOW"

                competition_color = "#2A9D8F"

                competition_message = (
                    "Competitive pressure around the searched location "
                    "is relatively limited, creating potential expansion "
                    "whitespace."
                )


            # --------------------------------------------------
            # 13. COMPETITION REPORT
            # --------------------------------------------------

            competition_report = f"""

            <style>

            .comp-report {{

                --bg:#FFFFFF;
                --card:#F7F8FA;
                --text:#172A46;
                --muted:#667085;
                --border:#E3E7ED;

                max-width:1150px;

                margin:28px auto;

                background:var(--bg);

                color:var(--text);

                border:
                    1px solid var(--border);

                border-radius:18px;

                overflow:hidden;

                font-family:
                    Inter,
                    -apple-system,
                    BlinkMacSystemFont,
                    "Segoe UI",
                    sans-serif;
            }}


            @media (prefers-color-scheme:dark) {{

                .comp-report {{

                    --bg:#191C21;
                    --card:#22262D;
                    --text:#F4F6F8;
                    --muted:#AAB2BF;
                    --border:#353B45;
                }}

            }}


            .comp-header {{

                padding:30px;

                border-bottom:
                    1px solid var(--border);
            }}


            .comp-eyebrow {{

                font-size:10px;

                letter-spacing:2px;

                font-weight:700;

                color:var(--muted);
            }}


            .comp-head-row {{

                display:flex;

                justify-content:
                    space-between;

                align-items:flex-end;

                gap:20px;

                margin-top:18px;
            }}


            .comp-title {{

                margin:0;

                font-size:28px;

                color:var(--text);
            }}


            .comp-pressure {{

                font-size:30px;

                font-weight:800;

                color:{competition_color};
            }}


            .comp-message {{

                margin-top:17px;

                max-width:850px;

                color:var(--muted);

                font-size:13px;

                line-height:1.65;
            }}


            .comp-grid {{

                display:grid;

                grid-template-columns:
                    repeat(4,1fr);

                gap:12px;

                padding:22px;
            }}


            .comp-card {{

                padding:18px;

                background:var(--card);

                border:
                    1px solid var(--border);

                border-radius:12px;
            }}


            .comp-label {{

                color:var(--muted);

                font-size:10px;

                font-weight:700;

                text-transform:uppercase;

                letter-spacing:.7px;
            }}


            .comp-value {{

                margin-top:8px;

                color:var(--text);

                font-size:25px;

                font-weight:750;
            }}


            .brand-section {{

                padding:
                    5px 24px 28px 24px;
            }}


            .brand-heading {{

                margin-bottom:18px;

                color:var(--muted);

                font-size:10px;

                font-weight:700;

                letter-spacing:1.5px;
            }}


            .brand-row {{

                display:grid;

                grid-template-columns:
                    170px 1fr 80px;

                align-items:center;

                gap:15px;

                margin:17px 0;
            }}


            .brand-name {{

                font-size:12px;

                font-weight:650;

                color:var(--text);
            }}


            .brand-track {{

                height:10px;

                background:var(--border);

                border-radius:100px;

                overflow:hidden;
            }}


            .brand-fill {{

                height:100%;

                border-radius:100px;
            }}


            .brand-stat {{

                text-align:right;

                color:var(--muted);

                font-size:11px;
            }}


            @media(max-width:750px) {{

                .comp-grid {{

                    grid-template-columns:
                        1fr 1fr;
                }}

                .comp-head-row {{

                    flex-direction:column;

                    align-items:flex-start;
                }}

            }}

            </style>


            <div class="comp-report">


                <div class="comp-header">


                    <div class="comp-eyebrow">

                        LOCAL COMPETITIVE INTELLIGENCE

                    </div>


                    <div class="comp-head-row">


                        <h2 class="comp-title">

                            {searched_location}
                            Competitive Landscape

                        </h2>


                        <div>


                            <div class="comp-pressure">

                                {competition_level}

                            </div>


                            <div style="
                                font-size:10px;
                                color:var(--muted);
                                text-align:right;
                            ">

                                COMPETITIVE PRESSURE

                            </div>


                        </div>


                    </div>


                    <div class="comp-message">

                        {competition_message}

                    </div>


                </div>


                <!-- COMPETITION KPIs -->

                <div class="comp-grid">


                    <div class="comp-card">

                        <div class="comp-label">

                            Nearby Stores

                        </div>


                        <div class="comp-value">

                            {total_stores}

                        </div>

                    </div>


                    <div class="comp-card">

                        <div class="comp-label">

                            Competitors

                        </div>


                        <div class="comp-value">

                            {competitor_stores}

                        </div>

                    </div>


                    <div class="comp-card">

                        <div class="comp-label">

                            {brand} Stores

                        </div>


                        <div class="comp-value">

                            {same_brand_stores}

                        </div>

                    </div>


                    <div class="comp-card">

                        <div class="comp-label">

                            Competitor Ratio

                        </div>


                        <div
                            class="comp-value"

                            style="
                                color:
                                {competition_color};
                            "
                        >

                            {competitor_ratio:.0%}

                        </div>

                    </div>


                </div>


                <!-- BRAND PRESENCE -->

                <div class="brand-section">


                    <div class="brand-heading">

                        BRAND PRESENCE WITHIN
                        {radius} KM OF
                        {searched_location.upper()}

                    </div>


                    <!-- BLINKIT -->

                    <div class="brand-row">


                        <div class="brand-name">

                            Blinkit

                        </div>


                        <div class="brand-track">

                            <div
                                class="brand-fill"

                                style="
                                    width:
                                    {blinkit_share:.1f}%;

                                    background:
                                    #F8CB46;
                                "
                            >
                            </div>

                        </div>


                        <div class="brand-stat">

                            {blinkit_count}

                            •
                            {blinkit_share:.0f}%

                        </div>


                    </div>


                    <!-- ZEPTO -->

                    <div class="brand-row">


                        <div class="brand-name">

                            Zepto

                        </div>


                        <div class="brand-track">

                            <div
                                class="brand-fill"

                                style="
                                    width:
                                    {zepto_share:.1f}%;

                                    background:
                                    #7A288A;
                                "
                            >
                            </div>

                        </div>


                        <div class="brand-stat">

                            {zepto_count}

                            •
                            {zepto_share:.0f}%

                        </div>


                    </div>


                    <!-- SWIGGY -->

                    <div class="brand-row">


                        <div class="brand-name">

                            Swiggy Instamart

                        </div>


                        <div class="brand-track">

                            <div
                                class="brand-fill"

                                style="
                                    width:
                                    {swiggy_share:.1f}%;

                                    background:
                                    #FC8019;
                                "
                            >
                            </div>

                        </div>


                        <div class="brand-stat">

                            {swiggy_count}

                            •
                            {swiggy_share:.0f}%

                        </div>


                    </div>


                </div>


            </div>

            """


            display(
                HTML(competition_report)
            )


            # --------------------------------------------------
            # 14. COMPETITION MAP
            # --------------------------------------------------

            display(
                HTML(
                    f"""
                    <div style="
                        max-width:1150px;
                        margin:32px auto 10px auto;
                        font-family:
                            Inter,
                            -apple-system,
                            BlinkMacSystemFont,
                            'Segoe UI',
                            sans-serif;
                    ">

                        <div style="
                            font-size:10px;
                            letter-spacing:2px;
                            opacity:.6;
                            font-weight:700;
                        ">

                            GEOSPATIAL VIEW

                        </div>


                        <div style="
                            font-size:23px;
                            font-weight:750;
                            margin-top:7px;
                        ">

                            {searched_location}
                            Local Dark-Store Network

                        </div>


                        <div style="
                            font-size:12px;
                            opacity:.6;
                            margin-top:5px;
                        ">

                            Actual searched coordinates:
                            {lat:.5f}, {lon:.5f}

                            • Search radius:
                            {radius} km

                        </div>

                    </div>
                    """
                )
            )


            fig = create_competition_map(
                lat,
                lon,
                nearby
            )

            fig.show()


        # ------------------------------------------------------
        # ERROR HANDLER
        # ------------------------------------------------------

        except Exception as e:

            display(
                HTML(
                    f"""
                    <div style="
                        max-width:1100px;
                        padding:18px;
                        margin:15px auto;
                        border:1px solid #E76F51;
                        border-radius:12px;
                    ">

                        <b>
                            Analysis Error
                        </b>

                        <br><br>

                        {type(e).__name__}:
                        {str(e)}

                    </div>
                    """
                )
            )


print(
    "✓ CELL 16 — FINAL ANALYSIS + REPORT READY"
)

✓ CELL 16 — FINAL ANALYSIS + REPORT READY


In [ ]:
analyse_button._click_handlers.callbacks = []

analyse_button.on_click(
    run_location_analysis
)

print("✓ ANALYSE LOCATION connected")

✓ ANALYSE LOCATION connected


In [ ]:
# FINAL INTERACTIVE APPLICATION

title = widgets.HTML(
    """
    <div style="margin:10px 0 18px 0;">

        <div style="
            font-size:24px;
            font-weight:700;
        ">
            Dark-Store Location Intelligence Engine
        </div>

        <div style="
            margin-top:5px;
            opacity:.65;
            font-size:13px;
        ">
            Search any Indian location or enter precise geographic
            coordinates to evaluate expansion suitability.
        </div>

    </div>
    """
)


place_box = widgets.VBox([

    widgets.HBox([
        place_input,
        search_button
    ]),

    search_results,

    search_status
])


coordinate_box = widgets.HBox([
    latitude_input,
    longitude_input
])


business_box = widgets.HBox([
    brand_input,
    radius_input
])


def change_input_mode(change):

    if change["new"] == "place":

        place_box.layout.display = ""
        coordinate_box.layout.display = "none"

    else:

        place_box.layout.display = "none"
        coordinate_box.layout.display = ""


input_mode.observe(
    change_input_mode,
    names="value"
)


coordinate_box.layout.display = "none"


display(title)

display(input_mode)

display(place_box)

display(coordinate_box)

display(business_box)

display(analyse_button)

display(output)

HTML(value='\n    <div style="margin:10px 0 18px 0;">\n\n        <div style="\n            font-size:24px;\n  …

ToggleButtons(description='Input:', options=(('Search Place', 'place'), ('Use Coordinates', 'coordinates')), v…

Button(button_style='success', description='ANALYSE LOCATION', layout=Layout(width='180px'), style=ButtonStyle…

Output()